# BioSkills Lab — Chapter 7
## Biological Data as Matrices and PCA on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> **Note:** The full experiment described in the Chapter 7 lesson uses the complete TCGA Pan-Cancer dataset (~500MB, 11,000 samples × 20,000 genes) which requires at least 32GB RAM. This notebook uses a smaller curated subset (750 samples × 160 genes via the cBioPortal API) that runs on **free-tier Google Colab**. To reproduce the full experiment, run this notebook on a local machine with sufficient RAM or on a compute cluster.

### What you will do
1. Download RNA-seq expression data for 5 cancer types from TCGA via the cBioPortal API
2. Represent the data as a samples × genes matrix
3. Run PCA to compress gene dimensions into 50 principal components
4. Visualise how well cancer types separate in PCA space

**The key question:** Does gene expression alone separate cancer types? And how many dimensions do you actually need?

### Install and import libraries
Standard data science stack. `requests` is used to query the cBioPortal REST API.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib requests

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
print('Libraries loaded')

### Download TCGA expression data via cBioPortal API

We query the **cBioPortal public API** for RNA-seq V2 RSEM expression values. The data covers:
- **BRCA** — breast cancer
- **LUAD** — lung adenocarcinoma
- **KIRC** — kidney renal clear cell carcinoma
- **PRAD** — prostate adenocarcinoma
- **COAD** — colon adenocarcinoma

We fetch 150 samples per cancer type and ~160 cancer-relevant genes. This keeps the total payload small enough for free Colab RAM.

**What to expect:** Each cancer fetch takes ~10–20 seconds. Total download ~2–3 minutes.

In [ ]:
BASE = 'https://www.cbioportal.org/api'
STUDIES = {'BRCA':'brca_tcga','LUAD':'luad_tcga','KIRC':'kirc_tcga','PRAD':'prad_tcga','COAD':'coad_tcga'}
GENE_IDS = [
    672, 675, 7157, 4609, 5728, 1029, 595, 896, 2064, 2065, 1956, 3320,
    4292, 5594, 5595, 5604, 5605, 207, 208, 10000, 4893, 4914, 5290, 6662,
    6663, 3845, 4763, 4764, 7422, 7424, 7525, 7528, 1499, 4005, 7409, 8503,
    9020, 2099, 2100, 2263, 2534, 3169, 3236, 4254, 4255, 5156, 5163, 5241,
    5579, 6009, 6196, 7163, 7490, 7849, 8379, 9054, 9500, 10257, 10319,
    10488, 11200, 23533, 51176, 55294, 57142, 64919, 1026, 1027, 1028, 1030,
    1031, 1032, 1869, 1870, 4616, 4619, 8317, 8318, 701, 994, 999, 1000,
    4193, 4831, 5925, 6237, 7153, 7538, 23135, 3551, 3553, 3569, 3586, 3588,
    3596, 6347, 6348, 6351, 6352, 6354, 6357, 2246, 2247, 2248, 2251, 2252,
    2253, 3908, 3909, 4318, 4320, 5265, 5266, 1278, 1280, 1281, 1282, 1284,
    1287, 1288, 1290, 3913, 3915, 4067, 4313, 6714, 6715, 6716, 6717, 6718,
    6720, 6721, 6722, 6723, 6724, 6725, 6726, 2977, 2978, 2982, 2983, 2984,
    2986, 2987, 2988, 2990, 1385, 1386, 4208, 4602, 5062, 5159, 7040, 7042,
    7043, 7046, 7048, 7049, 9369, 10912
]
print(f'Gene list: {len(GENE_IDS)} genes across 5 cancer types')

In [ ]:
def get_samples(study_id, n=150):
    r = requests.get(f'{BASE}/studies/{study_id}/samples', params={'pageSize': n}, timeout=30)
    r.raise_for_status()
    data = r.json()
    if not isinstance(data, list):
        raise ValueError(f'Unexpected response: {data}')
    return [s['sampleId'] for s in data[:n]]

def get_expression(study_id, sample_ids, gene_ids):
    r = requests.post(
        f'{BASE}/molecular-profiles/{study_id}_rna_seq_v2_mrna/molecular-data/fetch',
        params={'projection': 'SUMMARY'},
        json={'sampleIds': sample_ids, 'entrezGeneIds': gene_ids},
        timeout=120
    )
    return r.json() if r.status_code == 200 else None

In [ ]:
all_frames = []
for cancer, study in STUDIES.items():
    print(f'Fetching {cancer}...', end=' ')
    try:
        samples = get_samples(study, n=150)
        data    = get_expression(study, samples, GENE_IDS)
        if not data: print('no data'); continue
        df = pd.DataFrame([{'sample':d['sampleId'],'gene':d['entrezGeneId'],'value':d['value']}
                           for d in data if d.get('value') is not None])
        pivot = df.pivot_table(index='sample', columns='gene', values='value', aggfunc='first')
        pivot['cancer_type'] = cancer
        all_frames.append(pivot)
        print(f'{len(pivot)} samples, {pivot.shape[1]-1} genes')
    except Exception as e:
        print(f'failed: {e}')

if all_frames:
    expr = pd.concat(all_frames, axis=0)
    print(f'\nTotal: {expr.shape[0]} samples x {expr.shape[1]-1} genes')
else:
    print('API failed — using synthetic fallback')

### Synthetic fallback
If the API was unavailable, this cell generates data with similar structure: 5 cancer types, each with a distinct expression signature in a subset of genes. The PCA results will look similar to real TCGA data.

In [ ]:
if not all_frames:
    np.random.seed(42)
    n, g = 150, 500
    blocks, labels_list = [], []
    for i, ct in enumerate(STUDIES.keys()):
        X = np.random.randn(n, g) * 0.5
        X[:, i*100:(i+1)*100] += 3
        blocks.append(X); labels_list.extend([ct]*n)
    expr = pd.DataFrame(np.vstack(blocks), columns=[f'gene_{i}' for i in range(g)])
    expr['cancer_type'] = labels_list
    print(f'Synthetic data: {expr.shape}')

### Prepare the matrix

We extract the expression matrix `X` (samples x genes) and the label vector `y` (cancer type per sample). NaN values are replaced with 0.

**Note:** The cBioPortal RNA-seq V2 RSEM values are already log-transformed internally.

In [ ]:
y_raw = expr['cancer_type'].values
X_raw = np.nan_to_num(expr.drop(columns=['cancer_type']).values.astype(np.float32))
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
print(f'Matrix shape: {X_raw.shape[0]} samples x {X_raw.shape[1]} genes')
print(f'Cancer types: {le.classes_}')

### Train/test split and PCA

We split 80/20 **before** any scaling or PCA to prevent data leakage.

**StandardScaler** centres each gene to mean=0 and variance=1. **PCA** finds the directions of maximum variance and compresses to 50 components.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

n_comps = min(50, X_train_s.shape[1]-1, X_train_s.shape[0]-1)
pca = PCA(n_components=n_comps)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
print(f'PCA complete. Compressed to: {X_train_pca.shape}')

### Scree plot — how many PCs do you actually need?

The **scree plot** (left) shows variance per PC. The **cumulative variance plot** (right) shows when you have captured enough signal.

**How to interpret:** A steep drop after the first few PCs means strong low-dimensional structure. The red dashed line shows how many PCs explain 80% of variance.

In [ ]:
var = pca.explained_variance_ratio_
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar(range(1, len(var)+1), var*100)
plt.xlabel('Principal Component'); plt.ylabel('Variance Explained (%)')
plt.title('Scree Plot')
plt.subplot(1,2,2)
plt.plot(np.cumsum(var)*100, color='#38bdf8')
plt.xlabel('Number of PCs'); plt.ylabel('Cumulative Variance (%)')
plt.title('Cumulative Variance Explained')
plt.axhline(80, color='red', linestyle='--', label='80% threshold')
plt.legend()
plt.tight_layout(); plt.show()
print(f'PC1 explains: {var[0]*100:.1f}%')
print(f'Top 5 PCs explain: {sum(var[:5])*100:.1f}%')
print(f'Top 10 PCs explain: {sum(var[:10])*100:.1f}%')

### PCA scatter plot — do cancer types separate?

PC1 vs PC2, coloured by cancer type.

**How to interpret:** Tight, non-overlapping clusters mean cancer types have distinct expression profiles. Well-separated clusters in PC space also mean a linear classifier (logistic regression) will work well.

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))
colors = ['#f97316','#38bdf8','#4ade80','#a78bfa','#f43f5e']
for i, ct in enumerate(le.classes_):
    m = y_train == i
    ax.scatter(X_train_pca[m,0], X_train_pca[m,1],
               label=ct, alpha=0.6, s=20, color=colors[i % len(colors)])
ax.set_xlabel(f'PC1 ({var[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}% variance)')
ax.set_title('PCA of TCGA Pan-Cancer Gene Expression')
ax.legend(fontsize=9, title='Cancer type')
plt.tight_layout(); plt.show()
print('\nX_train_pca and X_test_pca are ready for Chapters 8, 9, and 10')